# ST 554 Analysis of BigData - Project 2
* Name: Yujin Kim
* Course: ST 554(601) Spring 2026 Analysis of Big Data
* Assignment: Project2

# Part I: Testing the SparkDataCheck Class
In this part of the project, I apply the custom `SparkDataCheck` class to the air quality dataset. The goal is to test each validation and summarization method using a Spark SQL DataFrame and confirm that the class behaves as intended under different conditions. I also include examples that produce warning message and examples with one sided bounds, as required in the project instructions.

## 1. Load Library and Class

In [1]:
#script import
import importlib
import spark_data_check
importlib.reload(spark_data_check)

<module 'spark_data_check' from '/home/jupyter-ykim68@ncsu.edu/Project2/spark_data_check.py'>

In [2]:
#Check file in notebook path
import os
print(os.listdir('/home/jupyter-ykim68@ncsu.edu/Project2'))

['__pycache__', 'weekly_nfl_data.csv', 'air.csv', 'Project2.ipynb', 'spark_data_check.py', '.ipynb_checkpoints']


In [3]:
#read data file and create object
from spark_data_check import SparkDataCheck
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()   #build SparkSession
obj = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")  #load air.csv file

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 16:18:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/26 16:18:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 2. Preliminary Check: Test Class #1
Instruction: Create a method that checks if each value in a numeric column is within user defined limits (upper and lower bounds, inclusive) and returns the dataframe with an appended column of Boolean values.

In [4]:
obj.df.show(5)

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-26 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-26 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-26 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-26 21:00:00|   2.2|       137

26/03/26 16:18:38 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ykim68@ncsu.edu/Project2/air.csv


In [5]:
print(obj.df.dtypes)

[('_c0', 'int'), ('Date', 'string'), ('Time', 'timestamp'), ('CO(GT)', 'double'), ('PT08.S1(CO)', 'int'), ('NMHC(GT)', 'int'), ('C6H6(GT)', 'double'), ('PT08.S2(NMHC)', 'int'), ('NOx(GT)', 'int'), ('PT08.S3(NOx)', 'int'), ('NO2(GT)', 'int'), ('PT08.S4(NO2)', 'int'), ('PT08.S5(O3)', 'int'), ('T', 'double'), ('RH', 'double'), ('AH', 'double')]


Before applying the method, I examined the data types of each column to identify which columns are numeric. The output shows that several columns, including `CO(GT)`, `T`, `RH`, and `AH`, are recognized as numeric types (double). Therefore, these columns are appropriate candidates for applying the numeric range validation method.

### Test: Numeric range method

I test the `check_numeric_range()` method to verify that it correctly evaluates whether values in a numeric column fall within user-defined bounds. 
This test ensures that the method properly handles valid numeric columns, applies inclusive bounds, and appends a Boolean column indicating whether each value satisfies the specified condition.

In [6]:
obj.check_numeric_range("T", lower=0, upper=40)
obj.df.select("T", "T_in_range").show(10)

+----+----------+
|   T|T_in_range|
+----+----------+
|13.6|      true|
|13.3|      true|
|11.9|      true|
|11.0|      true|
|11.2|      true|
|11.2|      true|
|11.3|      true|
|10.7|      true|
|10.7|      true|
|10.3|      true|
+----+----------+
only showing top 10 rows


After applying the `check_numeric_range()` method to the `T` column with lower and upper bounds of 0 and 40, a new Boolean column (`T_in_range`) was successfully sppended to the DataFrame. The results indicate that all displayed values fall within the specified range, and therefore the corresponding Boolean values are `True`.
The results confirm that the method correctly identifies whether values fall within the specified bounds, appends a Boolean column, and preserves the original DataFrame structure.

### Test: String level method
I test the `check_string_levels()` method to verify that it correctly evaluates whetehr values in a string column belong to a user-defined set of levels. This test ensures that the method properly identifies valid string values, handles non-matching values appropriately, and appends a Boolean column indicating whether each value satisfies the specified condition.

In [7]:
obj.check_string_levels("Date", ["10/03/2004"])
obj.df.select("Date", "Date_valid").show(10)

+---------+----------+
|     Date|Date_valid|
+---------+----------+
|3/10/2004|     false|
|3/10/2004|     false|
|3/10/2004|     false|
|3/10/2004|     false|
|3/10/2004|     false|
|3/10/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
+---------+----------+
only showing top 10 rows


After applying the `check_string_levels()` method to the `Date` column with the specified level `"10/03/2004"`, a new Boolean column (`Date_valid`) was successfully appended to the DataFrame. The result show that all displayed values are amrked as `False`, indicating that none of the values in the `Data` column match the specified level.
This outcome suggests that the actual data format in the dataset differs from the provided level.

In [8]:
obj.check_string_levels("Date", ["3/10/2004"])
obj.df.select("Date", "Date_valid").show(10)

+---------+----------+
|     Date|Date_valid|
+---------+----------+
|3/10/2004|      true|
|3/10/2004|      true|
|3/10/2004|      true|
|3/10/2004|      true|
|3/10/2004|      true|
|3/10/2004|      true|
|3/11/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
+---------+----------+
only showing top 10 rows


The dataset contains values such as `"3/10/2004"`, which do not match `"10/03/2004"` due to differences in formatting. The method correctly identifies that these values are not included in the specified set.

### Test: Missing Value Check Method
I test the `check_missing()` method to verify that it correctly identifies whether each value in a selected column is missing (`NULL`). This test ensures that the method properly appends a Boolean column indicating missingness without altering the original values in the DataFrame.

In [9]:
obj.check_missing("CO(GT)")
obj.df.select("CO(GT)", "CO(GT)_is_null").show(10)

+------+--------------+
|CO(GT)|CO(GT)_is_null|
+------+--------------+
|   2.6|         false|
|   2.0|         false|
|   2.2|         false|
|   2.2|         false|
|   1.6|         false|
|   1.2|         false|
|   1.2|         false|
|   1.0|         false|
|   0.9|         false|
|   0.6|         false|
+------+--------------+
only showing top 10 rows


After applying the `check_missing()` method to the `CO(GT)` column, a new Boolean column (`CO(GT)_is_null`) was successfully appended to the DataFrame. The displayed results show `False` for all visible rows, indicating that none of the values in this subset are missing.
This confirms that the method correctly detects missing values, appends the expected Boolean output column, and preserves the original DataFrame structure.

### Test: Min/Max Summarization Method
I test the `summarize_min_max()` method to verify that it correctly computes the minimum and maximum values of numeric columns. This test ensures that the method properly handles both individual column inputs and grouped summaries, and returns the results as a pandas DataFrame.

In [10]:
obj.summarize_min_max("T")

,T_min,T_max
0,-200.0,44.6


After applying the `summarize_min_max()` method to the `T` column, the results show that the minimum value is -200 and the maximum value 44.6. This indicates that the method correctly computes the summary statistics for the selected numeric column and returns the result as a pandas DataFrame. 
The output confirms that the method accurately extracts the minimum and maximum values without modifying the original DataFrame.

### Test: Count Summarization Method
I test the `summarize_counts()` method to verify that it correctly reports the frequency counts of levels in one or two string columns. This test ensure the method properly identifies valid string columns, computes counts for each level, and retruns the results as a pandas DataFrame.

In [11]:
obj.summarize_counts("Date")

,Date,count
0,9/2/2004,24
1,12/26/2004,24
2,2/18/2005,24
3,10/10/2004,24
4,10/11/2004,24
...,...,...
386,1/23/2005,24
387,6/28/2004,24
388,8/16/2004,24
389,12/20/2004,24


After applying the `summarize_counts()` method to the `Date` column, the results show the frequency count of each unique data in the dataset. Each row represents a distinct data value along with its corresponding count.
The output indicates that each data appears 24 times, which suggests that the dataset contains hourly observations for each day. This confirms that the method correctly groups the data by the specified string column, computes the counts for each level, and returns the results as a pandas DataFrame.

## 3. Example of Part1 Class
instruction: Provide 4-5 examples of using each of methods on this object. Show some examples where the messages need to print out, where only one bound is provided, etc.

In this section, I evaluated the `SparkDataCheck` class by applying all validation and summarization methods to the air quality dataset.
* The validation methods, including `check_numeric_range()`, `chec_string_levels()`, and `check_missing()`, correctly identified values based on the specified conditions. These methods successfully appended Boolean columns to the DataFrame without modifying the original data, demonstrating that they behave as intended.
* The summarization methods, `summarize_min_max()` and `summarize_counts()` accurately computed summary statistics and frequency counts. The results were returned as pandas DataFrames, as required, and provided meaningful insights into the dataset, such as the presence of placeholder values and the hourly structure of the data.

### A. Example of `check_numeric_range()`

In [12]:
#Example 1: lower and upper bounds
obj1 = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
obj1.check_numeric_range("T", lower=5, upper=20)
obj1.df.select("T", "T_in_range").show(10)

+----+----------+
|   T|T_in_range|
+----+----------+
|13.6|      true|
|13.3|      true|
|11.9|      true|
|11.0|      true|
|11.2|      true|
|11.2|      true|
|11.3|      true|
|10.7|      true|
|10.7|      true|
|10.3|      true|
+----+----------+
only showing top 10 rows


In [13]:
#Example 2: lower bound only
obj2 = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
obj2.check_numeric_range("AH", lower=0.5)
obj2.df.select("AH", "AH_in_range").show(10)

+------+-----------+
|    AH|AH_in_range|
+------+-----------+
|0.7578|       true|
|0.7255|       true|
|0.7502|       true|
|0.7867|       true|
|0.7888|       true|
|0.7848|       true|
|0.7603|       true|
|0.7702|       true|
|0.7648|       true|
|0.7517|       true|
+------+-----------+
only showing top 10 rows


In [14]:
#Example 3: upper bound only
obj3 = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
obj3.check_numeric_range("RH", upper=80)
obj3.df.select("RH", "RH_in_range").show(10)

+----+-----------+
|  RH|RH_in_range|
+----+-----------+
|48.9|       true|
|47.7|       true|
|54.0|       true|
|60.0|       true|
|59.6|       true|
|59.2|       true|
|56.8|       true|
|60.0|       true|
|59.7|       true|
|60.2|       true|
+----+-----------+
only showing top 10 rows


In [15]:
#Example 4: non-numeric column
obj4 = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
obj4.check_numeric_range("Date", lower=0, upper=10)

Column 'Date' is not numeric.


In [16]:
#Example 5: no bounds provided
obj5 = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
obj5.check_numeric_range("T")

At least one of lower or upper must be provided.


### B. Example of `check_string_levels()`

In [17]:
#Example 1: matching level
obj6 = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
obj6.check_string_levels("Date", ["3/10/2004"])
obj6.df.select("Date", "Date_valid").show(10)

+---------+----------+
|     Date|Date_valid|
+---------+----------+
|3/10/2004|      true|
|3/10/2004|      true|
|3/10/2004|      true|
|3/10/2004|      true|
|3/10/2004|      true|
|3/10/2004|      true|
|3/11/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
+---------+----------+
only showing top 10 rows


In [18]:
#Example 2: non-matching level
obj7 = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
obj7.check_string_levels("Date", ["10/03/2004"])
obj7.df.select("Date", "Date_valid").show(10)

+---------+----------+
|     Date|Date_valid|
+---------+----------+
|3/10/2004|     false|
|3/10/2004|     false|
|3/10/2004|     false|
|3/10/2004|     false|
|3/10/2004|     false|
|3/10/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
|3/11/2004|     false|
+---------+----------+
only showing top 10 rows


In [19]:
#Example 3: numeric column >> message
obj8 = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
obj8.check_string_levels("T", ["10", "20"])

Column 'T' is not a string column.


In [20]:
#Example 4: non-existing column
obj9 = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
obj9.check_string_levels("FakeColumn", ["A"])

Column 'FakeColumn' does not exist.


### C. Example of `check_missing()`

In [21]:
#Example 1
obj10 = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
obj10.check_missing("CO(GT)")
obj10.df.select("CO(GT)", "CO(GT)_is_null").show(10)

+------+--------------+
|CO(GT)|CO(GT)_is_null|
+------+--------------+
|   2.6|         false|
|   2.0|         false|
|   2.2|         false|
|   2.2|         false|
|   1.6|         false|
|   1.2|         false|
|   1.2|         false|
|   1.0|         false|
|   0.9|         false|
|   0.6|         false|
+------+--------------+
only showing top 10 rows


In [22]:
#Example 2
obj11 = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
obj11.check_missing("Date")
obj11.df.select("Date", "Date_is_null").show(10)

+---------+------------+
|     Date|Date_is_null|
+---------+------------+
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
+---------+------------+
only showing top 10 rows


In [23]:
#Example 3: non-existing column
obj12 = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
obj12.check_missing("NotAColumn")

Column 'NotAColumn' does not exist.


### D. Example of `summarize_min_max`

In [24]:
#Example 1: one numeric column
obj13 = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
obj13.summarize_min_max("T")

,T_min,T_max
0,-200.0,44.6


In [25]:
#Example 2: another_numeric column
obj13.summarize_min_max("RH")

,RH_min,RH_max
0,-200.0,88.7


In [26]:
#Example 3: non-numeric column
obj13.summarize_min_max("Date")

Column 'Date' is not numeric.


In [27]:
#Example 4: all numeric columns
obj13.summarize_min_max()

26/03/26 16:18:42 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: 
 Schema: _c0
Expected: _c0 but found: 
CSV file: file:///home/jupyter-ykim68@ncsu.edu/Project2/air.csv


,_c0_min,_c0_max,CO(GT)_min,CO(GT)_max,PT08.S1(CO)_min,PT08.S1(CO)_max,NMHC(GT)_min,NMHC(GT)_max,C6H6(GT)_min,C6H6(GT)_max,...,PT08.S4(NO2)_min,PT08.S4(NO2)_max,PT08.S5(O3)_min,PT08.S5(O3)_max,T_min,T_max,RH_min,RH_max,AH_min,AH_max
0,0,9356,-200.0,11.9,-200,2040,-200,1189,-200.0,63.7,...,-200,2775,-200,2523,-200.0,44.6,-200.0,88.7,-200.0,2.231


In [28]:
#Example 5: grouped summary
obj13.summarize_min_max("T", groupby="Date").head()

,Date,T_min,T_max
0,9/2/2004,21.0,38.7
1,12/26/2004,10.4,14.9
2,2/18/2005,5.4,11.8
3,10/10/2004,19.4,24.8
4,10/11/2004,17.9,23.2


### E. Example of `summarize_counts()`

In [29]:
#Example 1: valid string column
obj14 = SparkDataCheck.from_csv(spark, "/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
obj14.summarize_counts("Date")

,Date,count
0,9/2/2004,24
1,12/26/2004,24
2,2/18/2005,24
3,10/10/2004,24
4,10/11/2004,24
...,...,...
386,1/23/2005,24
387,6/28/2004,24
388,8/16/2004,24
389,12/20/2004,24


In [30]:
#Example 2: numeric column >>> message
obj14.summarize_counts("T")

Column 'T' is numeric.


In [31]:
#Example 3: non-existing column
obj14.summarize_counts("FakeColumn")

Column 'FakeColumn' does not exist.


Overall, the `SparkDataCheck` class demonstrate reliable and consistent performance across all implemented methods. The validation methods effectively identify data quality issues, while the summarization methods provide clear and interpretable statistical insights. 

## 4. Read that same data set in using `pandas`
Instruction: Now, read that same data set in using `pandas` (not `pandas-on-spark`). Use method to create an instance of this class from the `pandas` data frame.

In [32]:
import pandas as pd

air_pd = pd.read_csv("/home/jupyter-ykim68@ncsu.edu/Project2/air.csv")
air_pd.head()

,Unnamed: 0,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH
0,0,3/10/2004,18:00:00,2.6,1360,150,11.9,1046,166,1056,113,1692,1268,13.6,48.9,0.7578
1,1,3/10/2004,19:00:00,2.0,1292,112,9.4,955,103,1174,92,1559,972,13.3,47.7,0.7255
2,2,3/10/2004,20:00:00,2.2,1402,88,9.0,939,131,1140,114,1555,1074,11.9,54.0,0.7502
3,3,3/10/2004,21:00:00,2.2,1376,80,9.2,948,172,1092,122,1584,1203,11.0,60.0,0.7867
4,4,3/10/2004,22:00:00,1.6,1272,51,6.5,836,131,1205,116,1490,1110,11.2,59.6,0.7888


In [33]:
obj_pd = SparkDataCheck.from_pandas(spark, air_pd)
obj_pd.df.show(5)

+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|Unnamed: 0|     Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+----------+---------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|         0|3/10/2004|18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|         1|3/10/2004|19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|         2|3/10/2004|20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|         3|3/10/2004|21:00:00|   2.2|       1376|      80|     9.2|        

## 5. Example method
Instruction: Provide 1 example method call on that object.

In [34]:
obj_pd.check_missing("Date")
obj_pd.df.select("Date", "Date_is_null").show(10)

+---------+------------+
|     Date|Date_is_null|
+---------+------------+
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/10/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
|3/11/2004|       false|
+---------+------------+
only showing top 10 rows


The results confirm that the `SparkDataCheck` class functions correctly when initialized from a pandas DataFrame. The method produces the expected output, demonstrating that the class is compatible with different data input formats.

# Part II: NFL Data Analysis
In this part of the project, I analyze weekly NFL quarterback data using both pandas-on-Spark and Spark SQL DataFrames. The goal is to practice both APIs while summarizing quarterback passing performance across seasons. I focus on regular-season QB statistics from 2005 through 2023 and compare summary metrics such as completion percentage and touchdown-to-interception ratio.

## pandas-on-Spark

In [35]:
from pyspark.sql import SparkSession
import pyspark.pandas as ps

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


Before using pandas-on-Spark, I disabled ANSI mode in the Spark session because pandas-on-Spark may not operate properly when ANSI mode is enabled in this environment.

In [36]:
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.ansi.enabled", "false")
ps.set_option("compute.fail_on_ansi_mode", False)

### 1. Read NFL data
Instruction: Read in the weekly nfl data (csv file available at the project page, you’ll need to upload it to your JupyterHub)

In [37]:
nfl_ps = ps.read_csv("/home/jupyter-ykim68@ncsu.edu/Project2/weekly_nfl_data.csv")

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


### 2. Check out the DataFrame
Instruction: Check out the first 5 rows of the DataFrame

In [38]:
nfl_ps.head(5)

26/03/26 16:18:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


I first inspect the first five rows of the dataset to confirm that the file has been loaded correctly and to understand the basic structure of the variables.

## 3. Report all of the column names
Next, I examine the column names to identify which variables are needed for the QB performance analysis.

In [39]:
list(nfl_ps.columns)

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

## 4. QB stats for the seasons 05-23
**Instruction:** Subset the rows of the data to only include the position “QB”, the regular season (“REG”), and season in the range noted above

I filter the dataset to include only quaterbacks (`QB`), regular-season games (`REG`), and seasons from 2005 through 2023.

In [40]:
nfl_qb_ps = nfl_ps[(nfl_ps["position"] == "QB") & (nfl_ps["season_type"] == "REG") & (nfl_ps["season"] >= 2005) & (nfl_ps["season"] <= 2023)]

**Instruction:** Subset the columns to only include the `player_display_name`, `season`, `week`, `completions`, `attempts`, `passing_yards`, `passing_tds`, and `interceptions`

I then keep only the columns relevant to passing performance so that the analysis focuses on quarterback production and efficiency. 

In [41]:
nfl_qb_ps = nfl_qb_ps[["player_display_name", "season", "week", "completions", "attempts", "passing_yards", "passing_tds", "interceptions"]]

**Instruction:** For each `player_display_name` and `season combination`, fine the sum and mean of each of the statistical quantities (the rest of the columns we chose above)

I aggregate the weekly statistics by player and season. For each quaterback-season combination, I calculate both the sum and mean of the selected passing variables.

In [42]:
nfl_summary_ps = nfl_qb_ps.groupby(["player_display_name", "season"]).agg({"completions": ["sum", "mean"], 
                                                                           "attempts": ["sum", "mean"],
                                                                           "passing_yards": ["sum", "mean"],
                                                                           "passing_tds": ["sum", "mean"], 
                                                                           "interceptions": ["sum", "mean"]})

In [43]:
nfl_summary_ps.columns = [f"{col[0]}_{col[1]}" for col in nfl_summary_ps.columns] #make column flatten

nfl_summary_ps = nfl_summary_ps.reset_index()
print(nfl_summary_ps.columns)

Index(['player_display_name', 'season', 'completions_sum', 'completions_mean',
       'attempts_sum', 'attempts_mean', 'passing_yards_sum',
       'passing_yards_mean', 'passing_tds_sum', 'passing_tds_mean',
       'interceptions_sum', 'interceptions_mean'],
      dtype='object')


**Instruction:** Create two new variables (by season/player combination):
* completion_percentage = (sum of completions)/(sum of attempts)
* td_int_ratio = (sum passing tds)/(sum interceptions)

I then create two derived performance metrics. Completion percentage measures passing accuracy, while touchdown-to-interception ratio provides a simple indicator of scoring efficiency relative to turnovers.

In [44]:
nfl_summary_ps["completion_percentage"] = (nfl_summary_ps["completions_sum"] / nfl_summary_ps["attempts_sum"])

nfl_summary_ps["td_int_ratio"] = (nfl_summary_ps["passing_tds_sum"] / nfl_summary_ps["interceptions_sum"])

**Instroduction:** Subset the rows to only include player/season combinations wher ethe sum of attempts is at least 50.

To avoid misleading rnakings based on very small sample sizes, I keep only quarterback-season combinations with at least 50 pass attempts.

In [45]:
nfl_summary_ps_50 = nfl_summary_ps[nfl_summary_ps["attempts_sum"] >= 50]

**Instruction:** Sort the rows descending by completion_percentage and report the first 40 values!

I sort the filtered results by competion percentage in descending order to identify the most accurate quarterback-season performances.

In [46]:
completion_top40_ps = nfl_summary_ps_50.sort_values(by = "completion_percentage", ascending = False).head(40)
completion_top40_ps

,player_display_name,season,completions_sum,completions_mean,attempts_sum,attempts_mean,passing_yards_sum,passing_yards_mean,passing_tds_sum,passing_tds_mean,interceptions_sum,interceptions_mean,completion_percentage,td_int_ratio
1409,C.J. Beathard,2023,40,6.666667,53,8.833333,349.0,58.166667,1,0.166667,0.0,0.000000,0.754717,inf
1248,Colt McCoy,2021,74,10.571429,99,14.142857,740.0,105.714286,3,0.428571,1.0,0.142857,0.747475,3.000000
900,Matt Schaub,2019,50,10.000000,67,13.400000,580.0,116.000000,3,0.600000,1.0,0.200000,0.746269,3.000000
953,Drew Brees,2018,364,24.266667,489,32.600000,3992.0,266.133333,32,2.133333,5.0,0.333333,0.744376,6.400000
1057,Drew Brees,2019,281,25.545455,378,34.363636,2979.0,270.818182,27,2.454545,4.0,0.363636,0.743386,6.750000
1338,Mason Rudolph,2023,55,13.750000,74,18.500000,719.0,179.750000,3,0.750000,0.0,0.000000,0.743243,inf
1133,Taysom Hill,2020,88,5.500000,121,7.562500,928.0,58.000000,4,0.250000,2.0,0.125000,0.727273,2.000000
1007,Nick Foles,2018,141,28.200000,195,39.000000,1413.0,282.600000,7,1.400000,4.0,0.800000,0.723077,1.750000
917,Drew Brees,2017,386,24.125000,536,33.500000,4334.0,270.875000,23,1.437500,8.0,0.500000,0.720149,2.875000
851,Sam Bradford,2016,395,26.333333,552,36.800000,3877.0,258.466667,20,1.333333,5.0,0.333333,0.715580,4.000000


## Spark SQL DataFrame

**Instruction:** Now, repeat the above completely using the Spark SQL DataFrame. This includes reading in the data!

In [47]:
#check file name for fit my environment
import os
print(os.listdir('/home/jupyter-ykim68@ncsu.edu/Project2'))

from pyspark.sql import functions as F
import pyspark.pandas as ps

['__pycache__', 'weekly_nfl_data.csv', 'air.csv', 'Project2.ipynb', 'spark_data_check.py', '.ipynb_checkpoints']


In this section, I repeat the same NFL quarterback analysis using the Spark SQL DataFrame API. This allows me to compare the pandas-on-Spark worflow with the Spark SQL workflow while producing the same summary metrics and rankings.

### 1. Read Data

In [48]:
nfl_spark = spark.read.load('/home/jupyter-ykim68@ncsu.edu/Project2/weekly_nfl_data.csv', format="csv", header=True, inferSchema=True)

In [49]:
nfl_spark.show(5)

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sack_

### 2. Check column names

I first read the weekly NFL dataset as a Spark SQL DataFrame and inspect the first few rows and column names to confirm that the file has been loaded correctly.

In [50]:
print(nfl_spark.columns)

['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'recent_team', 'season', 'week', 'season_type', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share', 'wopr', 'special_teams_tds', 'fantasy_points', 'fantasy_points_ppr']


### 3. QB stats for the seasons 05-23

**Instruction:** Subset the rows of the data to only include the position “QB”, the regular season (“REG”), and
season in the range noted above

In [51]:
nfl_qb_spark = nfl_spark.filter((F.col("position") == "QB") & (F.col("season_type") == "REG") & (F.col("season") >= 2005) & (F.col("season") <= 2023))

**Instruction:** Subset the columns to only include the `player_display_name`, `season`, `week`, `completions`, `attempts`, `passing_yards`, `passing_tds`, and `interceptions`

I filter the dataset to include only regular-season quarterbacks from 2005 through 2023 and keep only the columns needed for the passing analysis.

In [52]:
nfl_qb_spark = nfl_qb_spark.select("player_display_name", "season", "week", "completions", "attempts", "passing_yards", "passing_tds", "interceptions")

**Instruction:** For each `player_display_name` and `season` combination, fine the sum and mean of each of the statistical quantities (the rest of the columns we chose above)

I group the data by quarterback and season and compute both the sum and mean of the selected passing statistics.

In [56]:
nfl_summary_spark = nfl_qb_spark.groupBy("player_display_name", "season").agg(
    F.sum("completions").alias("completions_sum"),
    F.mean("completions").alias("completions_mean"),
    F.sum("attempts").alias("attempts_sum"),
    F.mean("attempts").alias("attempts_mean"),
    F.sum("passing_yards").alias("passing_yards_sum"),
    F.mean("passing_yards").alias("passing_yards_mean"),
    F.sum("passing_tds").alias("passing_tds_sum"),
    F.mean("passing_tds").alias("passing_tds_mean"),
    F.sum("interceptions").alias("interceptions_sum"),
    F.mean("interceptions").alias("interceptions_mean"))

**Instruction:** Create two new variables (by season/player combination):
* completion_percentage = (sum of completions)/(sum of attempts)
* td_int_ratio = (sum passing tds)/(sum interceptions)

I create two derived metrics. These variables summarize quaterback accuracy and passing efficiency.

In [62]:
nfl_summary_spark = nfl_summary_spark.withColumn(
    "completion_percentage", F.col("completions_sum") / F.col("attempts_sum")).withColumn(
    "td_int_ratio", F.col("passing_tds_sum") / F.col("interceptions_sum"))

**Instruction:** Subset the rows to only include player/season combinations wher ethe sum of attempts is at least 50.

To avoid misleading results from very small sample sizes, I restrict the analysis to quaterback-season combinations with at least 50 pass attemps.

In [63]:
nfl_summary_spark_50 = nfl_summary_spark.filter(F.col("attempts_sum") >= 50)

**Instruction:** Sort the rows descending by completion_percentage and report the first 40 values!

I sort the filtered results by completion percentage in descending order to identify the most accurate quarterback-season performance.

In [64]:
completion_top40_spark = nfl_summary_spark_50.orderBy(F.col("completion_percentage").desc())
completion_top40_spark.show(40, truncate=False)

+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+-------------------+-----------------+-------------------+---------------------+------------------+------------------+
|player_display_name|season|completions_sum|completions_mean  |attempts_sum|attempts_mean     |passing_yards_sum|passing_yards_mean|passing_tds_sum|passing_tds_mean   |interceptions_sum|interceptions_mean |completion_percentage|td.int_ratio      |td_int_ratio      |
+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+-------------------+-----------------+-------------------+---------------------+------------------+------------------+
|C.J. Beathard      |2023  |40             |6.666666666666667 |53          |8.833333333333334 |349.0            |58.166666666666664|1              |0.16666666666666666|0.0              |0.0          

**Instruction:** Sort the rows descending by td_int_ratio and report the first 40 values!

I also sort the filtered results by touchdown-to-interception ratio to identify quarterback seasons with strong scoring efficiency relative to turnovers.

In [65]:
td_int_top40_spark = nfl_summary_spark_50.orderBy(F.col("td_int_ratio").desc())
td_int_top40_spark.show(40, truncate=False)

+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+-------------------+---------------------+-----------------+-----------------+
|player_display_name|season|completions_sum|completions_mean  |attempts_sum|attempts_mean     |passing_yards_sum|passing_yards_mean|passing_tds_sum|passing_tds_mean  |interceptions_sum|interceptions_mean |completion_percentage|td.int_ratio     |td_int_ratio     |
+-------------------+------+---------------+------------------+------------+------------------+-----------------+------------------+---------------+------------------+-----------------+-------------------+---------------------+-----------------+-----------------+
|Tom Brady          |2016  |291            |24.25             |432         |36.0              |3554.0           |296.1666666666667 |28             |2.3333333333333335|2.0              |0.16666666666666666|0.6